In [1]:
"""
Reproduce Table 1: Normalised Feature-Selection Scores
(↑ = stronger association with Aluminium)

11 bivariate dependence measures between Aluminium and each of
{Cu, Ni, Zn, Sn, Pb}, computed on log-returns, then min-max
normalised across the 5 metals so each row sums sensibly.
"""
!pip install dcor

import numpy as np
import pandas as pd
from scipy import stats, signal
import dcor
import pywt
from sklearn.feature_selection import mutual_info_regression
import warnings, itertools
warnings.filterwarnings("ignore")

# ── 0. Load & prepare data ──────────────────────────────────────────
df = pd.read_csv("Metals_Price.csv", parse_dates=["Date"])
df.set_index("Date", inplace=True)

# Log-returns (standard for financial dependence analysis)
ret = np.log(df / df.shift(1)).dropna()

target = ret["Aluminium"]
features = ["Copper", "Nickel", "Zinc", "Tin", "Lead"]
short    = ["Cu",     "Ni",     "Zn",   "Sn",  "Pb"]

# ── Helper: min-max normalise a dict of raw scores ──────────────────
def normalise(raw: dict) -> dict:
    vals = np.array(list(raw.values()))
    lo, hi = vals.min(), vals.max()
    if hi == lo:
        return {k: 1.0 for k in raw}
    return {k: (v - lo) / (hi - lo) for k, v in raw.items()}

# ── 1. Pearson correlation (on returns) ─────────────────────────────
def calc_pearson(target, ret, features):
    raw = {}
    for f in features:
        r, _ = stats.pearsonr(target, ret[f])
        raw[f] = abs(r)
    return normalise(raw)

# ── 2. Spearman rank correlation ────────────────────────────────────
def calc_spearman(target, ret, features):
    raw = {}
    for f in features:
        r, _ = stats.spearmanr(target, ret[f])
        raw[f] = abs(r)
    return normalise(raw)

# ── 3. Kendall τ ────────────────────────────────────────────────────
def calc_kendall(target, ret, features):
    raw = {}
    for f in features:
        r, _ = stats.kendalltau(target, ret[f])
        raw[f] = abs(r)
    return normalise(raw)

# ── 4. Distance Correlation (dCor) ──────────────────────────────────
def calc_dcor(target, ret, features):
    raw = {}
    t = target.values
    for f in features:
        raw[f] = dcor.distance_correlation(t, ret[f].values)
    return normalise(raw)

# ── 5. HSIC (Hilbert-Schmidt Independence Criterion) ────────────────
def rbf_kernel_matrix(x, sigma=None):
    x = x.reshape(-1, 1)
    sq_dist = np.sum((x[:, None, :] - x[None, :, :]) ** 2, axis=-1)
    if sigma is None:
        sigma = np.median(np.sqrt(sq_dist[sq_dist > 0]))
    return np.exp(-sq_dist / (2 * sigma ** 2))

def hsic(x, y):
    n = len(x)
    K = rbf_kernel_matrix(x)
    L = rbf_kernel_matrix(y)
    H = np.eye(n) - np.ones((n, n)) / n
    return np.trace(K @ H @ L @ H) / (n - 1) ** 2

def calc_hsic(target, ret, features):
    raw = {}
    t = target.values
    for f in features:
        raw[f] = hsic(t, ret[f].values)
    return normalise(raw)

# ── 6. Mutual Information (kNN-based) ───────────────────────────────
def calc_mi(target, ret, features):
    raw = {}
    t = target.values
    for f in features:
        # Average over multiple random seeds for stability
        mi_vals = []
        for rs in range(5):
            mi = mutual_info_regression(
                ret[f].values.reshape(-1, 1), t,
                n_neighbors=7, random_state=rs
            )[0]
            mi_vals.append(mi)
        raw[f] = np.mean(mi_vals)
    return normalise(raw)

# ── 7. Transfer Entropy  (binned estimator, feature → Al) ──────────
def transfer_entropy(source, target, bins=10, lag=1):
    """TE from source → target using histogram estimator."""
    t_now  = target[lag:]
    t_past = target[:-lag]
    s_past = source[:-lag]

    # Discretise
    t_now_d  = np.digitize(t_now,  np.linspace(t_now.min(),  t_now.max(),  bins))
    t_past_d = np.digitize(t_past, np.linspace(t_past.min(), t_past.max(), bins))
    s_past_d = np.digitize(s_past, np.linspace(s_past.min(), s_past.max(), bins))

    # Joint counts
    def entropy(*arrays):
        combined = np.column_stack(arrays)
        _, counts = np.unique(combined, axis=0, return_counts=True)
        p = counts / counts.sum()
        return -np.sum(p * np.log2(p + 1e-12))

    # TE = H(t_now, t_past) + H(t_past, s_past) - H(t_now, t_past, s_past) - H(t_past)
    te = (entropy(t_now_d, t_past_d)
          + entropy(t_past_d, s_past_d)
          - entropy(t_now_d, t_past_d, s_past_d)
          - entropy(t_past_d))
    return max(te, 0.0)

def calc_te(target, ret, features):
    raw = {}
    t = target.values
    for f in features:
        raw[f] = transfer_entropy(ret[f].values, t)
    return normalise(raw)

# ── 8. Tail Dependence (empirical, lower + upper averaged) ──────────
def tail_dependence(x, y, q=0.05):
    """Empirical tail dependence coefficient (average of lower & upper)."""
    n = len(x)
    # Convert to pseudo-observations (ranks / (n+1))
    u = stats.rankdata(x) / (n + 1)
    v = stats.rankdata(y) / (n + 1)

    # Lower tail: P(V <= q | U <= q)
    mask_lower = u <= q
    if mask_lower.sum() > 0:
        lambda_l = np.mean(v[mask_lower] <= q)
    else:
        lambda_l = 0.0

    # Upper tail: P(V > 1-q | U > 1-q)
    mask_upper = u >= (1 - q)
    if mask_upper.sum() > 0:
        lambda_u = np.mean(v[mask_upper] >= (1 - q))
    else:
        lambda_u = 0.0

    return (lambda_l + lambda_u) / 2

def calc_tail(target, ret, features):
    raw = {}
    t = target.values
    for f in features:
        raw[f] = tail_dependence(t, ret[f].values)
    return normalise(raw)

# ── 9. Wavelet Coherence (D5 detail, ~32-day scale) ─────────────────
def wavelet_corr(x, y, level=5):
    """Correlation of DWT detail coefficients at specified level."""
    
    # 🔑 FIX: ensure writable arrays
    x = np.asarray(x).copy()
    y = np.asarray(y).copy()

    min_len = min(len(x), len(y))
    max_level = pywt.dwt_max_level(min_len, 'db4')
    use_level = min(level, max_level)

    cx = pywt.wavedec(x, 'db4', level=use_level)
    cy = pywt.wavedec(y, 'db4', level=use_level)

    dx = cx[1]
    dy = cy[1]

    r, _ = stats.pearsonr(dx, dy)
    return abs(r)

def calc_wavelet(target, ret, features):
    raw = {}
    t = target.values
    for f in features:
        raw[f] = wavelet_corr(t, ret[f].values)
    return normalise(raw)

# ── 10. Spectral Coherence (mean coherence across frequencies) ──────
def spectral_coherence(x, y, nperseg=256):
    f, Cxy = signal.coherence(x, y, nperseg=min(nperseg, len(x)//4))
    return np.mean(Cxy)

def calc_spectral(target, ret, features):
    raw = {}
    t = target.values
    for f in features:
        raw[f] = spectral_coherence(t, ret[f].values)
    return normalise(raw)

# ── 11. Lagged Partial Correlation ──────────────────────────────────
def partial_corr_lagged(target_series, feature_series, other_features_df, max_lag=5):
    """
    Max absolute partial correlation of feature(t-k) with target(t),
    controlling for all other metals' returns at lag k.
    """
    best = 0.0
    for lag in range(1, max_lag + 1):
        # Align data
        y = target_series.iloc[lag:].values
        x = feature_series.iloc[:-lag].values
        # Controls: other features at the same lag
        Z = other_features_df.iloc[:-lag].values

        # Partial corr via residuals
        # Regress y on Z
        Z_with_const = np.column_stack([Z, np.ones(len(Z))])
        try:
            beta_y = np.linalg.lstsq(Z_with_const, y, rcond=None)[0]
            res_y = y - Z_with_const @ beta_y

            beta_x = np.linalg.lstsq(Z_with_const, x, rcond=None)[0]
            res_x = x - Z_with_const @ beta_x

            r, _ = stats.pearsonr(res_y, res_x)
            best = max(best, abs(r))
        except:
            pass
    return best

def calc_lag_partial(target, ret, features):
    raw = {}
    for f in features:
        others = [c for c in features if c != f]
        raw[f] = partial_corr_lagged(target, ret[f], ret[others])
    return normalise(raw)


# ═══════════════════════════════════════════════════════════════════════
#  MAIN: Compute all methods, build table
# ═══════════════════════════════════════════════════════════════════════
print("Computing 11 feature-selection criteria …\n")

methods = {
    "Pearson (ret.)":   calc_pearson(target, ret, features),
    "Spearman":         calc_spearman(target, ret, features),
    "Kendall τ":        calc_kendall(target, ret, features),
    "dCor":             calc_dcor(target, ret, features),
    "HSIC":             calc_hsic(target, ret, features),
    "Mutual Info.":     calc_mi(target, ret, features),
    "Transfer Ent.":    calc_te(target, ret, features),
    "Tail Dep.":        calc_tail(target, ret, features),
    "Wavelet (D5)":     calc_wavelet(target, ret, features),
    "Spectral Coh.":    calc_spectral(target, ret, features),
    "Lag. Partial":     calc_lag_partial(target, ret, features),
}

# Build DataFrame
rows = []
for method, scores in methods.items():
    row = {"Method": method}
    for f, s in zip(features, short):
        row[s] = scores[f]
    rows.append(row)

table = pd.DataFrame(rows).set_index("Method")

# Mean and Rank rows
means = table.mean()
ranks = means.rank(ascending=False).astype(int)
rank_labels = {1: "1st", 2: "2nd", 3: "3rd"}

table.loc["Mean"] = means
table.loc["Rank"] = ranks

# ── Pretty-print ────────────────────────────────────────────────────
print("=" * 62)
print("  Normalised Feature-Selection Scores (↑ = stronger w/ Al)")
print("=" * 62)
print(table.iloc[:-1].round(3).to_string())  # methods + mean
print("-" * 62)
rank_str = "  ".join(f"{s}: {rank_labels.get(int(ranks[s]), str(int(ranks[s]))+'th')}"
                     for s in short)
print(f"Rank  →  {rank_str}")
print("=" * 62)

# Also save to CSV
table.round(3).to_csv("feature_selection_results.csv")
print("\nResults saved to feature_selection_results.csv")

Computing 11 feature-selection criteria …

  Normalised Feature-Selection Scores (↑ = stronger w/ Al)
                   Cu     Ni     Zn     Sn     Pb
Method                                           
Pearson (ret.)  1.000  0.585  0.883  0.000  0.494
Spearman        1.000  0.467  0.844  0.000  0.389
Kendall τ       1.000  0.449  0.844  0.000  0.375
dCor            1.000  0.466  0.849  0.000  0.388
HSIC            1.000  0.418  0.893  0.000  0.327
Mutual Info.    1.000  0.382  0.778  0.000  0.333
Transfer Ent.   0.990  0.154  1.000  0.000  0.550
Tail Dep.       1.000  0.449  0.957  0.000  0.377
Wavelet (D5)    0.762  0.564  1.000  0.540  0.000
Spectral Coh.   1.000  0.539  0.909  0.000  0.460
Lag. Partial    1.000  0.906  0.118  0.000  0.635
Mean            0.977  0.489  0.825  0.049  0.393
--------------------------------------------------------------
Rank  →  Cu: 1st  Ni: 3rd  Zn: 2nd  Sn: 5th  Pb: 4th

Results saved to feature_selection_results.csv
